# AMOS 22（Zenodo）→ Google Drive

在 Colab 中挂载 Google Drive，从 Zenodo 记录 [7262581](https://zenodo.org/records/7262581) 下载 **AMOS 2022** 相关文件到 `dataset/pretrain/Amos/download/`：

- `amos22.zip`：解压见下一格（输出在与 `download` 平级的 `unzip/`）
- `labeled_data_meta_0000_0599.csv`：元数据表，无需解压

下载使用 `wget -c` 与 `--show-progress`，支持断点续传；失败项会在末尾汇总。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess

DEST = "/content/drive/MyDrive/dataset/pretrain/Amos/download"
os.makedirs(DEST, exist_ok=True)

WGET_CMD = ["wget", "-c", "--show-progress"]

URLS = [
    (
        "https://zenodo.org/records/7262581/files/amos22.zip?download=1",
        "amos22.zip",
    ),
    (
        "https://zenodo.org/records/7262581/files/labeled_data_meta_0000_0599.csv?download=1",
        "labeled_data_meta_0000_0599.csv",
    ),
]

n = len(URLS)
failed = []

for i, (url, filename) in enumerate(URLS, start=1):
    out_path = os.path.join(DEST, filename)
    print(f"\n[{i}/{n}] {filename}")
    print(f"    -> {out_path}")
    print(f"    {url}")
    try:
        subprocess.run([*WGET_CMD, "-O", out_path, url], check=True)
        print("    OK")
    except subprocess.CalledProcessError as e:
        failed.append((filename, url, e.returncode))
        print(f"    FAILED (exit {e.returncode})")

print("\n" + "=" * 60)
if failed:
    print(f"下载失败 {len(failed)} 个（共 {n} 个）：")
    for filename, url, code in failed:
        print(f"  - {filename}  exit={code}")
        print(f"    {url}")
    raise RuntimeError(f"{len(failed)} 个文件下载失败，见上方列表")
print(f"全部完成（{n} 个）。目录内容：")
print(sorted(os.listdir(DEST)))

In [ ]:
# 解压：仅处理 download 下的 .zip；每个 zip 解压到 unzip/<与压缩包同名的目录>/
import os
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/Amos/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/Amos/unzip"
os.makedirs(UNZIP_ROOT, exist_ok=True)

zip_names = sorted(f for f in os.listdir(DOWNLOAD) if f.lower().endswith(".zip"))
failed = []

for i, zname in enumerate(zip_names, start=1):
    zpath = os.path.join(DOWNLOAD, zname)
    out_dir = os.path.join(UNZIP_ROOT, os.path.splitext(zname)[0])
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n[{i}/{len(zip_names)}] {zname}")
    print(f"    -> {out_dir}")
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise zipfile.BadZipFile(f"损坏条目: {bad}")
            zf.extractall(out_dir)
        print("    OK")
    except Exception as e:
        failed.append((zname, str(e)))
        print(f"    FAILED: {e}")

print("\n" + "=" * 60)
if failed:
    print(f"解压失败 {len(failed)} 个：")
    for zname, err in failed:
        print(f"  - {zname}: {err}")
    raise RuntimeError("部分 zip 解压失败，见上方列表")
print("全部 zip 解压完成。")
print("unzip 下顶层：", sorted(os.listdir(UNZIP_ROOT)))